In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# made my me, I just had AI define DATA_DIR and OUTPUT_DIR
import pandas as pd
import os


DATA_DIR   = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"

os.makedirs(OUTPUT_DIR, exist_ok=True)

OSError: [Errno 30] Read-only file system: '/kaggle'

In [ ]:
#All made by me.
def base_filter(df):
    if "season" in df.columns:
        df = df[df["season"] >= 2000].copy()
    if "lg" in df.columns:
        df = df[df["lg"] == "NBA"].copy()
        df.drop(columns=["lg"], inplace=True)
    return df


#All made by me.
def fill_numeric(df):
    num = df.select_dtypes(include="number").columns
    df[num] = df[num].fillna(0)
    return df

#All made by me.
def fill_numeric(df):
    num = df.select_dtypes(include="number").columns
    df[num] = df[num].fillna(0)
    return df

#All made by me.
def merge_new_cols(base, other, on):
    new_cols = [c for c in other.columns if c not in base.columns or c in on]
    return base.merge(other[new_cols], on=on, how="left")

#All made by me.
def merge_new_cols(base, other, on):
    new_cols = [c for c in other.columns if c not in base.columns or c in on]
    return base.merge(other[new_cols], on=on, how="left")

#All made by me.
def find_file(name):
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            if filename == name:
                return os.path.join(dirname, filename)
    raise FileNotFoundError(f"Could not find {name}")

# I had AI do this part (too advanced for me)
df = pd.read_csv(find_file("Player Per Game.csv"))
df = base_filter(df)
df = df[df["g"] >= 10]
df.drop(columns=["x2p_per_game","x2pa_per_game","x2p_percent","e_fg_percent"], errors="ignore", inplace=True)
era = df["season"] >= 1980
for col in ["x3p_per_game","x3pa_per_game","x3p_percent"]:
    if col in df.columns: df.loc[era, col] = df.loc[era, col].fillna(0)
df["gs"] = df["gs"].fillna(-1)
df["pos"] = df["pos"].fillna("UNK")
for col in ["tov_per_game","stl_per_game","blk_per_game","orb_per_game","drb_per_game"]:
    if col in df.columns: df[col] = df[col].fillna(0)
df.dropna(subset=["pts_per_game","fg_percent","mp_per_game"], inplace=True)
df.sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/player_per_game.csv", index=False)
print("player_per_game.csv done")

df = pd.read_csv(find_file("Team Stats Per Game.csv"))
df = base_filter(df)
df.drop(columns=["x2p_per_game","x2pa_per_game","x2p_percent","mp_per_game"], errors="ignore", inplace=True)
df["playoffs"] = df["playoffs"].astype(int)
df = fill_numeric(df)
df.sort_values(["season","team"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/team_per_game.csv", index=False)
print("team_per_game.csv done")

df = pd.read_csv(find_file("Team Stats Per 100 Poss.csv"))
df = base_filter(df)
df.drop(columns=["x2p_per_100_poss","x2pa_per_100_poss","x2p_percent","mp"], errors="ignore", inplace=True)
df["playoffs"] = df["playoffs"].astype(int)
df = fill_numeric(df)
df.sort_values(["season","team"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/team_per_100.csv", index=False)
print("team_per_100.csv done")

df = pd.read_csv(find_file("Opponent Stats Per Game.csv"))
df = base_filter(df)
df.drop(columns=["opp_x2p_per_game","opp_x2pa_per_game","opp_x2p_percent","mp_per_game"], errors="ignore", inplace=True)
df["playoffs"] = df["playoffs"].astype(int)
df = fill_numeric(df)
df.sort_values(["season","team"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/opponent_per_game.csv", index=False)
print("opponent_per_game.csv done")

#Mostly done by me, with corrections from AI. 
df = pd.read_csv(find_file("Player Career Info.csv"))
df.drop(columns=["birth_date","colleges","debut"], errors="ignore", inplace=True)
df.rename(columns={"from":"career_start","to":"career_end","ht_in_in":"height_in","wt":"weight_lbs"}, inplace=True)
df["hof"] = df["hof"].fillna(False).astype(int)
df["pos"] = df["pos"].fillna("UNK")
df["height_in"] = df["height_in"].fillna(df["height_in"].median())
df["weight_lbs"] = df["weight_lbs"].fillna(df["weight_lbs"].median())
df.drop_duplicates(subset="player_id").reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/player_career_info.csv", index=False)
print("player_career_info.csv done")

#Mostly done by me, with corrections from AI. 
df = pd.read_csv(find_file("Player Season Info.csv"))
df = base_filter(df)
df = df[["season","player","player_id","team","experience"]].copy()
df["experience"] = df["experience"].replace("R", 0)
df["experience"] = pd.to_numeric(df["experience"], errors="coerce").fillna(-1).astype(int)
df.drop_duplicates(subset=["player_id","season","team"]).sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/player_season_info.csv", index=False)
print("player_season_info.csv done")

# Award Shares
df = pd.read_csv(find_file("Player Award Shares.csv"))
df = df[df["season"] >= 2000].copy()
keep_awards = ["nba mvp","dpoy","roy","mip","smoy","6moy","cpoy","finals mvp"]
df = df[df["award"].str.lower().isin(keep_awards)]
df["award"] = df["award"].str.lower().str.replace(" ","_")
df = df.pivot_table(index=["season","player","player_id"], columns="award", values="share", aggfunc="max").reset_index().fillna(0)
df.columns.name = None
df.sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/player_award_shares.csv", index=False)
print("player_award_shares.csv done")

# Award Shares
df = pd.read_csv(find_file("Player Award Shares.csv"))
df = df[df["season"] >= 2000].copy()
keep_awards = ["nba mvp","dpoy","roy","mip","smoy","6moy","cpoy","finals mvp"]
df = df[df["award"].str.lower().isin(keep_awards)]
df["award"] = df["award"].str.lower().str.replace(" ","_")
df = df.pivot_table(index=["season","player","player_id"], columns="award", values="share", aggfunc="max").reset_index().fillna(0)
df.columns.name = None
df.sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/player_award_shares.csv", index=False)
print("player_award_shares.csv done")

#Mostly done by me, with corrections from AI. 
df = pd.read_csv(find_file("End of Season Teams.csv"))
df = base_filter(df)
df = df[df["type"].str.contains("NBA|Defensive", case=False, na=False)].copy()
df["is_all_nba"] = df["type"].str.contains("NBA", case=False).astype(int)
df["is_all_defense"] = df["type"].str.contains("Defensive", case=False).astype(int)
df["team_number"] = df["number_tm"].astype(str).str.extract(r"(\d+)")[0].fillna(0).astype(int)
df = df[["season","player","player_id","position","type","is_all_nba","is_all_defense","team_number"]]
df.drop_duplicates().sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/end_of_season_teams.csv", index=False)
print("end_of_season_teams.csv done")

#All made by me.
df = pd.read_csv(find_file("All-Star Selections.csv"))
df = base_filter(df)
df["is_allstar"] = 1
df.drop(columns=["replaced"], errors="ignore", inplace=True)
df.drop_duplicates(subset=["player_id","season"]).sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/allstar_selections.csv", index=False)
print("allstar_selections.csv done")

import os
for f in os.listdir("/kaggle/working"):
    print(f)

# I had AI do this part (too advanced for me)
PLAYER_KEY = ["player_id","season","team"]
ppg  = pd.read_csv(f"{OUTPUT_DIR}/player_per_game.csv")
adv  = pd.read_csv(f"{OUTPUT_DIR}/advanced.csv")
p100 = pd.read_csv(f"{OUTPUT_DIR}/per_100_poss.csv")
pbp  = pd.read_csv(f"{OUTPUT_DIR}/player_pbp.csv")
psh  = pd.read_csv(f"{OUTPUT_DIR}/player_shooting.csv")
ci   = pd.read_csv(f"{OUTPUT_DIR}/player_career_info.csv")
si   = pd.read_csv(f"{OUTPUT_DIR}/player_season_info.csv")
aw   = pd.read_csv(f"{OUTPUT_DIR}/player_award_shares.csv")
ast  = pd.read_csv(f"{OUTPUT_DIR}/allstar_selections.csv")
eos  = pd.read_csv(f"{OUTPUT_DIR}/end_of_season_teams.csv")
master = ppg.copy()
for other in [adv, p100, pbp, psh]:
    new_cols = [c for c in other.columns if c not in master.columns or c in PLAYER_KEY]
    master = master.merge(other[new_cols], on=PLAYER_KEY, how="left")
master = master.merge(ci[["player_id","height_in","weight_lbs","career_start","career_end","hof"]], on="player_id", how="left")
master = master.merge(si[["player_id","season","team","experience"]], on=PLAYER_KEY, how="left")
new_cols = [c for c in aw.columns if c not in master.columns or c in ["player_id","season"]]
master = master.merge(aw[new_cols], on=["player_id","season"], how="left")
master = master.merge(ast[["player_id","season","is_allstar"]], on=["player_id","season"], how="left")
eos_agg = eos.groupby(["player_id","season"]).agg(is_all_nba=("is_all_nba","max"), is_all_defense=("is_all_defense","max"), best_all_nba_team=("team_number","min")).reset_index()
master = master.merge(eos_agg, on=["player_id","season"], how="left")
for col in ["is_allstar","is_all_nba","is_all_defense","best_all_nba_team"]:
    master[col] = master[col].fillna(0).astype(int)
master["experience"] = master["experience"].fillna(-1).astype(int)
master["hof"] = master["hof"].fillna(0).astype(int)
master["height_in"] = master["height_in"].fillna(master["height_in"].median())
master["weight_lbs"] = master["weight_lbs"].fillna(master["weight_lbs"].median())
master.sort_values(["season","player"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/master_player.csv", index=False)
print("master_player.csv done")

#I had AI do this part (too advanced for me)
TEAM_KEY = ["abbreviation","season"]
ts   = pd.read_csv(f"{OUTPUT_DIR}/team_summaries.csv")
tpg  = pd.read_csv(f"{OUTPUT_DIR}/team_per_game.csv")
t100 = pd.read_csv(f"{OUTPUT_DIR}/team_per_100.csv")
opp  = pd.read_csv(f"{OUTPUT_DIR}/opponent_per_game.csv")
mteam = ts.copy()
for other in [tpg, t100, opp]:
    new_cols = [c for c in other.columns if c not in mteam.columns or c in TEAM_KEY]
    mteam = mteam.merge(other[new_cols], on=TEAM_KEY, how="left")
num_cols = mteam.select_dtypes(include="number").columns
mteam[num_cols] = mteam[num_cols].fillna(0)
mteam.sort_values(["season","team"]).reset_index(drop=True).to_csv(f"{OUTPUT_DIR}/master_team.csv", index=False)
print("master_team.csv done")

print("\nALL DONE — files saved to /kaggle/working/")